In [1]:
import numpy as np
import pandas as pd

# optional deps
try:
    from sklearn.covariance import LedoitWolf, OAS
    HAVE_SKLEARN = True
except Exception:
    HAVE_SKLEARN = False

try:
    import cvxpy as cp
    HAVE_CVXPY = True
except Exception:
    HAVE_CVXPY = False

try:
    from scipy.optimize import minimize
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False


def shrink_cov(cov: np.ndarray, method: str = "none", data: np.ndarray | None = None) -> np.ndarray:
    """
    method: 'none' | 'ridge' | 'diag_shrink' | 'ledoit_wolf' | 'oas'
    data: (T x N) returns (LW/OAS用)
    """
    method = method.lower()
    cov = 0.5 * (cov + cov.T)
    n = cov.shape[0]

    if method == "none":
        return cov

    if method == "ridge":
        avg_var = float(np.trace(cov) / n)
        eps = 1e-6 * max(avg_var, 1e-12)
        return cov + eps * np.eye(n)

    if method == "diag_shrink":
        alpha = 0.1
        diag = np.diag(np.diag(cov))
        return (1 - alpha) * cov + alpha * diag

    if method in ("ledoit_wolf", "oas"):
        if (not HAVE_SKLEARN) or (data is None):
            # fallback
            alpha = 0.1
            diag = np.diag(np.diag(cov))
            return (1 - alpha) * cov + alpha * diag
        est = LedoitWolf().fit(data) if method == "ledoit_wolf" else OAS().fit(data)
        return est.covariance_

    raise ValueError(f"unknown shrink method: {method}")


def cov_metrics(cov: np.ndarray) -> dict:
    cov = 0.5 * (cov + cov.T)
    evals = np.linalg.eigvalsh(cov)
    eigmin = float(evals.min())
    eigmax = float(evals.max())
    cond = float("inf") if eigmin <= 0 else float(eigmax / eigmin)
    return {"eigmin": eigmin, "eigmax": eigmax, "cond": cond}


def corr_metrics(cov: np.ndarray) -> dict:
    d = np.sqrt(np.diag(cov))
    corr = cov / np.outer(d, d)
    np.fill_diagonal(corr, 1.0)
    n = corr.shape[0]
    off = corr[~np.eye(n, dtype=bool)]
    return {
        "corr_mean_offdiag": float(np.mean(off)),
        "corr_median_offdiag": float(np.median(off)),
        "corr_max_offdiag": float(np.max(off)),
        "corr_min_offdiag": float(np.min(off)),
    }


def mv_weights_longonly_sum1(cov: np.ndarray) -> np.ndarray:
    """
    min w'Cov w s.t. w>=0, sum(w)=1
    """
    n = cov.shape[0]
    cov = 0.5 * (cov + cov.T)

    if HAVE_CVXPY:
        w = cp.Variable(n)
        prob = cp.Problem(cp.Minimize(cp.quad_form(w, cov)), [w >= 0, cp.sum(w) == 1])
        try:
            prob.solve(solver=cp.OSQP, verbose=False)
        except Exception:
            prob.solve(solver=cp.SCS, verbose=False)
        wv = np.array(w.value).reshape(-1)
        if wv is None or np.any(~np.isfinite(wv)):
            raise RuntimeError(f"cvxpy failed: {prob.status}")
        wv = np.clip(wv, 0, None)
        wv /= wv.sum()
        return wv

    if HAVE_SCIPY:
        x0 = np.ones(n) / n
        bounds = [(0.0, 1.0) for _ in range(n)]
        cons = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0},)

        def f(w):  return float(w @ cov @ w)
        def g(w):  return 2.0 * (cov @ w)

        res = minimize(f, x0, jac=g, bounds=bounds, constraints=cons, method="SLSQP",
                       options={"ftol": 1e-12, "maxiter": 10000, "disp": False})
        if not res.success:
            raise RuntimeError(f"SLSQP failed: {res.message}")
        wv = np.clip(res.x, 0, None)
        wv /= wv.sum()
        return wv

    raise RuntimeError("Need cvxpy or scipy to solve MV.")


def weight_metrics(w: np.ndarray) -> dict:
    w = np.asarray(w, float)
    w = np.clip(w, 0, None)
    w /= w.sum()
    entropy = float(-np.sum(np.where(w > 0, w * np.log(w), 0.0)))
    return {
        "w_max": float(w.max()),
        "w_hhi": float(np.sum(w * w)),        # concentration (1/N〜1)
        "w_entropy": entropy,                # diversification-ish
        "w_nnz_1e-6": float(np.sum(w > 1e-6)),
    }


def bootstrap_stability(R: np.ndarray, shrink: str, B: int = 60, frac: float = 0.7, seed: int = 0) -> dict:
    """
    R: (T x N) returns
    ブートストラップで MV weights を何回も解いて「重みのブレ」を測る
    """
    rng = np.random.default_rng(seed)
    T, N = R.shape
    k = max(50, int(frac * T))

    Ws = []
    fails = 0
    for _ in range(B):
        idx = rng.integers(0, T, size=k)  # iid bootstrap
        Rb = R[idx, :]
        cov = np.cov(Rb, rowvar=False, ddof=1)
        cov = shrink_cov(cov, method=shrink, data=Rb)
        try:
            w = mv_weights_longonly_sum1(cov)
            Ws.append(w)
        except Exception:
            fails += 1

    if len(Ws) == 0:
        return {"boot_fail_rate": 1.0, "boot_w_std_mean": np.nan, "boot_w_std_max": np.nan}

    W = np.vstack(Ws)
    stds = W.std(axis=0, ddof=1) if W.shape[0] > 1 else np.zeros(N)
    return {
        "boot_fail_rate": fails / B,
        "boot_w_std_mean": float(np.mean(stds)),
        "boot_w_std_max": float(np.max(stds)),
    }


def evaluate_candidate_sets(
    returns_df: pd.DataFrame,
    candidates: dict[str, list[str]],
    *,
    date_start: str | None = None,
    date_end: str | None = None,
    ann: int = 252,
    shrink: str = "none",
    bootstrap_B: int = 60,
    bootstrap_frac: float = 0.7,
    seed: int = 0,
) -> pd.DataFrame:
    sub = returns_df.copy()
    if date_start is not None:
        sub = sub.loc[pd.to_datetime(date_start):]
    if date_end is not None:
        sub = sub.loc[:pd.to_datetime(date_end)]

    rows = []
    for name, cols in candidates.items():
        missing = [c for c in cols if c not in sub.columns]
        if missing:
            rows.append({"set": name, "error": f"missing: {missing}", "assets": ", ".join(cols)})
            continue

        X = sub[cols].dropna()
        T, N = X.shape
        if T < 300:
            rows.append({"set": name, "error": f"too few rows: {T}", "assets": ", ".join(cols)})
            continue

        R = X.to_numpy(float)
        cov = np.cov(R, rowvar=False, ddof=1)
        cov = shrink_cov(cov, method=shrink, data=R)
        cov_ann = cov * ann

        row = {"set": name, "N": N, "T": T, "shrink": shrink, "assets": ", ".join(cols)}
        row.update(corr_metrics(cov))
        row.update(cov_metrics(cov_ann))

        try:
            w = mv_weights_longonly_sum1(cov_ann)
            row.update(weight_metrics(w))
            row.update(bootstrap_stability(R, shrink=shrink, B=bootstrap_B, frac=bootstrap_frac, seed=seed))
            row["error"] = ""
        except Exception as e:
            row["error"] = str(e)

        rows.append(row)

    res = pd.DataFrame(rows)
    # 見やすい順に並べ替え：壊れてない → 条件数小 → 相関小 → ブートストラップ安定
    res = res.sort_values(by=["error", "cond", "corr_mean_offdiag", "boot_w_std_mean"],
                          ascending=[True, True, True, True])
    return res


In [5]:
import pandas as pd

rets = pd.read_pickle("asset_data/asset_returns.pkl")
rets.index = pd.to_datetime(rets.index)
rets = rets.sort_index()

candidates = {
    "equity_5": ["LargeCap","MidCap","SmallCap","EAFE","EM"],
    "set1": ["LargeCap","REIT","EAFE","HighYield","Gold"],
    "set2": ["LargeCap","REIT","EM","HighYield","Gold"],
    "set3": ["LargeCap","REIT","EM","Corporate","Gold"],
    "set4": ["LargeCap","Commodity","EM","Corporate","Gold"],
    "set5": ["LargeCap","Commodity","EM","HighYield","Gold"],

}

res = evaluate_candidate_sets(rets, candidates, shrink="ledoit_wolf")
print(res)


/var/folders/27/3yxrjnr94l12ymwg7m474pph0000gn/T/ipykernel_28673/3413169475.py:125: RuntimeWarning: divide by zero encountered in log
  entropy = float(-np.sum(np.where(w > 0, w * np.log(w), 0.0)))
/var/folders/27/3yxrjnr94l12ymwg7m474pph0000gn/T/ipykernel_28673/3413169475.py:125: RuntimeWarning: invalid value encountered in multiply
  entropy = float(-np.sum(np.where(w > 0, w * np.log(w), 0.0)))
/var/folders/27/3yxrjnr94l12ymwg7m474pph0000gn/T/ipykernel_28673/3413169475.py:125: RuntimeWarning: divide by zero encountered in log
  entropy = float(-np.sum(np.where(w > 0, w * np.log(w), 0.0)))
/var/folders/27/3yxrjnr94l12ymwg7m474pph0000gn/T/ipykernel_28673/3413169475.py:125: RuntimeWarning: invalid value encountered in multiply
  entropy = float(-np.sum(np.where(w > 0, w * np.log(w), 0.0)))


        set  N     T       shrink                                    assets  \
4      set4  5  8808  ledoit_wolf  LargeCap, Commodity, EM, Corporate, Gold   
5      set5  5  8836  ledoit_wolf  LargeCap, Commodity, EM, HighYield, Gold   
3      set3  5  8564  ledoit_wolf       LargeCap, REIT, EM, Corporate, Gold   
2      set2  5  8564  ledoit_wolf       LargeCap, REIT, EM, HighYield, Gold   
1      set1  5  8564  ledoit_wolf     LargeCap, REIT, EAFE, HighYield, Gold   
0  equity_5  5  8312  ledoit_wolf      LargeCap, MidCap, SmallCap, EAFE, EM   

   corr_mean_offdiag  corr_median_offdiag  corr_max_offdiag  corr_min_offdiag  \
4           0.111030             0.090563          0.411001         -0.079529   
5           0.188489             0.198838          0.410531         -0.015730   
3           0.155719             0.046421          0.677460         -0.040998   
2           0.234081             0.238317          0.677435         -0.014668   
1           0.240182             0.238732

/var/folders/27/3yxrjnr94l12ymwg7m474pph0000gn/T/ipykernel_28673/3413169475.py:125: RuntimeWarning: divide by zero encountered in log
  entropy = float(-np.sum(np.where(w > 0, w * np.log(w), 0.0)))
/var/folders/27/3yxrjnr94l12ymwg7m474pph0000gn/T/ipykernel_28673/3413169475.py:125: RuntimeWarning: invalid value encountered in multiply
  entropy = float(-np.sum(np.where(w > 0, w * np.log(w), 0.0)))
